# RQ2 — Statistical Tests: Resolution Strategies

This notebook runs the formal statistical tests supporting the RQ2 results:

1. **Wilson CIs** for global strategy proportions (V1, V2, CC, CB, NC, NN)
2. **Chi-squared goodness-of-fit** against Ghiotto et al.'s published distribution
3. **Chi-squared test of independence**: strategy × agent
4. **Cramér's V** effect size for strategy × agent
5. **Chi-squared test of independence**: Imprecise rate × agent (localization quality)
6. **Post-hoc pairwise chi-squared** for agent pairs on the strategy distribution
7. **Chi-squared test of independence**: strategy × resolver (preview of RQ3)

Required packages: `scipy`, `statsmodels`.  Both are in the project venv.

In [ ]:
import sys
from pathlib import Path
from itertools import combinations

_here = Path.cwd()
for candidate in [_here, *_here.parents]:
    if (candidate / 'analysis' / 'common.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests

from analysis.common import (
    load_tables, build_chunk_frame,
    STRATEGY_ORDER, _RAW_TO_CANONICAL,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('Imports OK')

## 1. Load data and build classifiable chunk frame

In [ ]:
tables = load_tables()
chunks = build_chunk_frame(tables)

# Map raw strategy labels to canonical ones (V1, V2, CC, CB, NC, NN, Imprecise)
chunks['strategy_canon'] = chunks['strategy'].map(_RAW_TO_CANONICAL).fillna('Imprecise')

STRATEGIES_CLASSIFIABLE = ['V1', 'V2', 'CC', 'CB', 'NC', 'NN']

# Classifiable = not Imprecise
classifiable = chunks[chunks['strategy_canon'] != 'Imprecise'].copy()

n_total    = len(chunks)
n_classif  = len(classifiable)
n_imprecise = n_total - n_classif

print(f'Total chunks:       {n_total:,}')
print(f'Classifiable:       {n_classif:,}  ({n_classif/n_total:.1%})')
print(f'Imprecise:          {n_imprecise:,}  ({n_imprecise/n_total:.1%})')
print()
print('Global strategy distribution (classifiable only):')
print(classifiable['strategy_canon'].value_counts().reindex(STRATEGIES_CLASSIFIABLE))

## 2. Wilson CIs for global strategy proportions

For each classifiable strategy s: proportion = count(s) / n_classifiable.
Wilson (score) interval at 95% confidence.

In [ ]:
counts = classifiable['strategy_canon'].value_counts().reindex(STRATEGIES_CLASSIFIABLE, fill_value=0)

rows = []
for strat, cnt in counts.items():
    lo, hi = proportion_confint(cnt, n_classif, alpha=0.05, method='wilson')
    rows.append({
        'strategy': strat,
        'count': cnt,
        'proportion': cnt / n_classif,
        'ci_lo': lo,
        'ci_hi': hi,
    })

wilson_df = pd.DataFrame(rows)
wilson_df['pct'] = wilson_df['proportion'].map('{:.1%}'.format)
wilson_df['ci'] = wilson_df.apply(lambda r: f'[{r.ci_lo:.1%}, {r.ci_hi:.1%}]', axis=1)
print(wilson_df[['strategy', 'count', 'pct', 'ci']].to_string(index=False))

## 3. Chi-squared goodness-of-fit against Ghiotto et al.'s distribution

H₀: Our strategy distribution is consistent with the one reported by
Ghiotto et al. (2020) for human Java merges.

**Caveat**: This is a cross-study comparison (different populations — Java vs.
multi-language, humans vs. agents). A significant result does NOT imply
the populations differ in a meaningful way; it merely indicates that the
observed distribution is unlikely if the null were true at our sample size.
With n ≈ 60k, even tiny differences will be significant. Report alongside
the raw proportions and Wilson CIs for honest interpretation.

Ghiotto proportions (Table 4 of their paper, classifiable only):
  V1: 0.425, V2: 0.145, CC: 0.066, CB: 0.074, NC: 0.218, NN: 0.072
(Source: Ghiotto et al., TSE 2020, Table 4 — values normalised to sum to 1
over the six classifiable categories, excluding their Imprecise.)

In [ ]:
# Ghiotto published proportions (classifiable only, normalised to sum to 1)
# Adjust these values if you have more precise numbers from their paper
GHIOTTO = {
    'V1': 0.425,
    'V2': 0.145,
    'CC': 0.066,
    'CB': 0.074,
    'NC': 0.218,
    'NN': 0.072,
}

observed = counts.reindex(STRATEGIES_CLASSIFIABLE).values.astype(float)
expected_props = np.array([GHIOTTO[s] for s in STRATEGIES_CLASSIFIABLE])
expected_props /= expected_props.sum()  # ensure sums to 1
expected_counts = expected_props * n_classif

chi2_stat, gof_p = chisquare(f_obs=observed, f_exp=expected_counts)

print('Chi-squared goodness-of-fit (our distribution vs Ghiotto baseline):')
print(f'  χ²({len(STRATEGIES_CLASSIFIABLE)-1}) = {chi2_stat:.1f},  p = {gof_p:.2e}')
print()
print('Comparison table:')
comp = pd.DataFrame({
    'strategy':   STRATEGIES_CLASSIFIABLE,
    'ours_pct':   [f'{observed[i]/n_classif:.1%}' for i in range(len(STRATEGIES_CLASSIFIABLE))],
    'ghiotto_pct':[f'{expected_props[i]:.1%}'    for i in range(len(STRATEGIES_CLASSIFIABLE))],
})
print(comp.to_string(index=False))
print()
print('NOTE: At n≈60k, any non-trivial difference will yield p≪0.05.')
print('The Wilson CIs above are more informative for magnitude assessment.')

## 4. Chi-squared test of independence: strategy × agent

H₀: The strategy distribution is the same across all coding agents.

Expected-count assumption: all cells ≥ 5 (will be verified below).

In [ ]:
contingency_agent = (
    classifiable
    .groupby(['agent', 'strategy_canon'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=STRATEGIES_CLASSIFIABLE, fill_value=0)
)

print('Contingency table (strategy × agent):')
print(contingency_agent.to_string())

chi2, p, dof, expected = chi2_contingency(contingency_agent.values)

print(f'\nChi-squared test of independence:')
print(f'  χ²({dof}) = {chi2:.1f},  p = {p:.2e}')

min_expected = expected.min()
pct_lt5 = (expected < 5).sum() / expected.size * 100
print(f'  Min expected count: {min_expected:.1f}  |  Cells < 5: {pct_lt5:.1f}%')
if pct_lt5 > 20:
    print('  WARNING: >20% of cells have expected count < 5; chi-squared approximation may be unreliable.')
    print('  Consider collapsing low-frequency cells or using Fisher exact via Monte Carlo.')

## 5. Cramér's V effect size (strategy × agent)

Cramér's V ∈ [0, 1]. For a 6×5 table (df_min = min(rows-1, cols-1) = 4):
V < 0.10 → weak; V ∈ [0.10, 0.30) → moderate; V ≥ 0.30 → strong.

In [ ]:
def cramers_v(chi2_val, n, r, c):
    """Bias-corrected Cramér's V (Bergsma 2013)."""
    phi2 = chi2_val / n
    phi2_corr = max(0, phi2 - (r - 1) * (c - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    c_corr = c - (c - 1) ** 2 / (n - 1)
    return np.sqrt(phi2_corr / min(r_corr - 1, c_corr - 1))

r, c = contingency_agent.shape
V = cramers_v(chi2, n_classif, r, c)
print(f"Cramér's V (bias-corrected) = {V:.3f}")

if V < 0.10:
    label = 'weak'
elif V < 0.30:
    label = 'moderate'
else:
    label = 'strong'
print(f'Effect size: {label}')

## 6. Imprecise rate × agent (localization quality check)

H₀: The probability that a chunk is Imprecise is the same across all agents.

This test verifies whether localization failures are uniformly distributed,
or whether some agents produce merges that are inherently harder to localize.

In [ ]:
imprecise_contingency = (
    chunks
    .assign(is_imprecise=(chunks['strategy_canon'] == 'Imprecise').astype(int))
    .groupby(['agent', 'is_imprecise'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: 'classifiable', 1: 'imprecise'})
)
imprecise_contingency['imprecise_rate'] = (
    imprecise_contingency['imprecise'] /
    (imprecise_contingency['classifiable'] + imprecise_contingency['imprecise'])
).map('{:.1%}'.format)

print('Imprecise rate by agent:')
print(imprecise_contingency.to_string())

chi2_imp, p_imp, dof_imp, _ = chi2_contingency(
    imprecise_contingency[['classifiable', 'imprecise']].values
)
print(f'\nChi-squared:  χ²({dof_imp}) = {chi2_imp:.1f},  p = {p_imp:.2e}')

## 7. Post-hoc pairwise chi-squared between agents

If strategy × agent is significant, identify which agent pairs drive the effect.
p-values are corrected with Holm-Bonferroni (more powerful than full Bonferroni).

In [ ]:
agents = contingency_agent.index.tolist()
pairs = list(combinations(agents, 2))

raw_ps = []
for a1, a2 in pairs:
    sub = contingency_agent.loc[[a1, a2]].values
    # Only run if both rows have sufficient counts
    if sub.sum() < 10:
        raw_ps.append(1.0)
        continue
    _, p_pair, _, _ = chi2_contingency(sub)
    raw_ps.append(p_pair)

_, corrected_ps, _, _ = multipletests(raw_ps, method='holm')

pairwise_df = pd.DataFrame({
    'agent_1': [p[0] for p in pairs],
    'agent_2': [p[1] for p in pairs],
    'p_raw':   [f'{p:.4f}' for p in raw_ps],
    'p_holm':  [f'{p:.4f}' for p in corrected_ps],
    'significant': ['*' if p < 0.05 else '' for p in corrected_ps],
})
print(pairwise_df.to_string(index=False))

## 8. Chi-squared test of independence: strategy × resolver

H₀: The strategy distribution is the same for agent-resolved and human-resolved chunks.

This is the key test underpinning the 86.6% vs 45.2% V1 contrast reported
in Sections 4.2 and 4.3 of the paper. The full analysis lives in
rq3_statistical_tests.ipynb; this cell provides a preview.

In [ ]:
if 'resolver_type' not in classifiable.columns:
    print('Column resolver_type not found — run rq3_statistical_tests.ipynb for this analysis.')
else:
    contingency_resolver = (
        classifiable[classifiable['resolver_type'].isin(['agent', 'human'])]
        .groupby(['resolver_type', 'strategy_canon'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=STRATEGIES_CLASSIFIABLE, fill_value=0)
    )
    print(contingency_resolver.to_string())

    chi2_res, p_res, dof_res, exp_res = chi2_contingency(contingency_resolver.values)
    print(f'\nChi-squared:  χ²({dof_res}) = {chi2_res:.1f},  p = {p_res:.2e}')

    r_res, c_res = contingency_resolver.shape
    V_res = cramers_v(chi2_res, contingency_resolver.values.sum(), r_res, c_res)
    print(f"Cramér's V = {V_res:.3f}")

## 9. Summary

In [ ]:
print('=== RQ2 STATISTICAL TEST SUMMARY ===')
print()
print('(1) Wilson CIs for global strategy proportions — see wilson_df above')
print(f'(2) GoF vs Ghiotto: χ²({len(STRATEGIES_CLASSIFIABLE)-1}) = {chi2_stat:.1f}, p = {gof_p:.2e}')
print(f'(3) Independence strategy × agent: χ²({dof}) = {chi2:.1f}, p = {p:.2e}')
print(f"(4) Cramér's V (strategy × agent) = {V:.3f} ({label} effect)")
print(f'(5) Imprecise rate × agent: χ²({dof_imp}) = {chi2_imp:.1f}, p = {p_imp:.2e}')
print('(6) Pairwise Holm-corrected p-values — see pairwise_df above')
print('(7) strategy × resolver — see rq3_statistical_tests.ipynb for full analysis')